# Goal and Work Summary

**Goal**: Build and evaluate a full-image semantic segmentation pipeline using **YOLOv8-seg** (instance segmentation model adapted for dense per-pixel predictions) to detect defects in green rough oak planks (Background, BlackRot, Knot, Stain). This notebook uses the native YOLO training loop with configurable hyperparameters, optional transfer learning, and comprehensive test-set evaluation.

## Workflow Stages and Rationale

**Stage 1 - Data audit and class normalization**
- Scan raw line-scan data and normalize class naming.
- Use reusable helpers in `notebooks/functions`.

**Why this choice**: Consistent labels improve training stability and metric reliability.

**Stage 2 - YOLO-format dataset preparation**
- Convert organized-data splits to YOLO folder structure (images, labels, data.yaml).
- Build YOLO-compatible segmentation mask annotations.

**Why this choice**: YOLO expects this structure; native `.train()` API uses it directly.

**Stage 3 - Model training with native YOLO API**
- Load YOLOv8-seg model (nano/small/medium configurable).
- Train with YOLO's `.train()` method—built-in LR scheduling, augmentation, validation.
- Optional backbone freezing for warmup epochs, then fine-tune.

**Why this choice**: YOLO's training loop is optimized, stable, and fully handles hyperparameter scheduling.

**Stage 4 - Transfer learning (optional)**
- Load prior checkpoint if available (e.g., from another oak dataset or generic YOLO-seg).
- Fine-tune on target dataset.

**Why this choice**: Domain adaptation and faster convergence when prior knowledge exists.

**Stage 5 - Test-set evaluation**
- Reload best checkpoint and compute validation metrics.
- Per-class mAP50, mAP50-95, IoU, precision, recall, F1.

**Why this choice**: Multi-metric view captures different strengths/weaknesses.

**Stage 6 - Results logging**
- Save best checkpoint to deployment-ready location.
- Log all metrics to CSV/JSON for tracking.

**Why this choice**: Easy tracking across experiments; .pt file ready for Docker inference.

## References (Libraries)

| Library | Purpose | Link |
| --- | --- | --- |
| ultralytics | YOLOv8 model and training API | https://github.com/ultralytics/ultralytics |
| PyTorch | Underlying deep learning framework | https://pytorch.org/ |
| Albumentations | Image augmentations and normalization | https://albumentations.ai/ |
| scikit-learn | Evaluation metrics | https://scikit-learn.org/ |

In [ ]:
# Prepare dataset + core imports
from functions.data_pipeline import prepare_dataset
from functions.oak_hf_dataset import ensure_oak_raw_data
from ultralytics import YOLO
import torch
from pathlib import Path
import json
import yaml
import numpy as np
from PIL import Image
import shutil
import cv2
from datetime import datetime

# Change this to use a custom folder name under data/
RAW_FOLDER_NAME = "raw-data"
RAW_DIR = ensure_oak_raw_data(raw_folder_name=RAW_FOLDER_NAME)

# Prepare dataset and get class mapping
data_root, class_mapping = prepare_dataset(
    raw_data_folder=str(RAW_DIR),
    mask_threshold=350,
    val_split=0.16,
    test_split=0.2,
 )

print(f"Data root: {data_root}")
print(f"Class mapping: {class_mapping}")

PREPARE_DATASET — SKIPPED (data already prepared)
  Organized data: ../data/organized-data
  Classes: {'BlackRot': 1, 'Knot': 2, 'Stain': 3}
  (pass force=True to re-run the full pipeline)
Data root: ../data/organized-data
Class mapping: {'BlackRot': 1, 'Knot': 2, 'Stain': 3}


In [2]:
# ──────────────────────────────────────────────
# Configuration - change these knobs as needed
# ──────────────────────────────────────────────

GPU_ID = 0  # choose 0 or 1

if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
    DEVICE = torch.device(f"cuda:{GPU_ID}")
    print(f"Using GPU {GPU_ID}: {torch.cuda.get_device_name(GPU_ID)}")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

# YOLO model configuration
YOLO_MODEL = "yolov8l-seg"  # nano (n), small (s), medium (m), large (l), xlarge (x)
IMAGE_HEIGHT = 384
IMAGE_WIDTH = 1024
TRAIN_IMGSZ = max(IMAGE_HEIGHT, IMAGE_WIDTH)  # YOLO train/val expects integer imgsz
BATCH_SIZE = 16  # AutoBatch: let YOLO pick a safe batch size for available VRAM
NUM_WORKERS = 2  
NUM_CLASSES = len(class_mapping) + 1  # +1 for Background

# Remap disk mask class ids → training ids (leave {} for no remap).
# Example: alphabetical folders Knot=1, Rot=2 vs old BlackRot=1, Knot=2 → use {1: 2, 2: 1, 3: 3}
MASK_ID_REMAP = {}

# ──────────────────────────────────────────────
# Training configuration
# ──────────────────────────────────────────────
LEARNING_RATE = 1e-3
NUM_EPOCHS = 100
PATIENCE = 20  # early stopping patience

# Optional: freeze backbone for warmup epochs
FREEZE_BACKBONE = False
FREEZE_EPOCHS = 5

# ──────────────────────────────────────────────
# Transfer learning (fine-tuning) configuration
# ──────────────────────────────────────────────
# Set True when adapting a trained model to a new scanner/domain.
TRANSFER_LEARNING = False

# Path to a prior checkpoint saved by this repo.
# Example: "../models/YOLO_SegHead_FullImage/best.pt"
TRANSFER_CHECKPOINT_PATH = None

# ──────────────────────────────────────────────
# Paths and naming
# ──────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
YOLO_DATASET_ROOT = PROJECT_ROOT / "data" / "yolo-format"
MODEL_DIR = PROJECT_ROOT / "models" / "STACK_YOLO_L_FullImage"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics"
TENSORBOARD_DIR = PROJECT_ROOT / "logs" / "tensorboard" / "Stack_YOLO_L_FullImage"

for d in [MODEL_DIR, METRICS_DIR, TENSORBOARD_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Create timestamp for unique runs
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
METRICS_RUN_NAME = f"stack_yolo_l_fullimage_{RUN_TIMESTAMP}"

print(f"Project root: {PROJECT_ROOT}")
print(f"YOLO dataset root: {YOLO_DATASET_ROOT}")
print(f"Model dir: {MODEL_DIR}")
print(f"Metrics run name: {METRICS_RUN_NAME}")
print(f"Training image size (int): {TRAIN_IMGSZ}")

Using GPU 0: NVIDIA GeForce RTX 3090
Project root: /workspaces/Oak-Defect-Detection
YOLO dataset root: /workspaces/Oak-Defect-Detection/data/yolo-format
Model dir: /workspaces/Oak-Defect-Detection/models/STACK_YOLO_L_FullImage
Metrics run name: stack_yolo_l_fullimage_20260722_164922
Training image size (int): 1024


In [3]:
# Build class names by TRAINING id (after MASK_ID_REMAP), not disk id.
inv_train = {}
for name, disk_id in class_mapping.items():
    train_id = int(MASK_ID_REMAP.get(disk_id, disk_id)) if MASK_ID_REMAP else int(disk_id)
    if train_id in inv_train:
        raise ValueError(
            f"MASK_ID_REMAP collision: train_id={train_id} maps to both "
            f"{inv_train[train_id]} and {name}"
        )
    inv_train[train_id] = name

missing_train_ids = [i for i in range(1, NUM_CLASSES) if i not in inv_train]
if missing_train_ids:
    raise ValueError(
        f"Missing class names for training ids {missing_train_ids}. "
        "Check class_mapping and MASK_ID_REMAP."
    )

CLASS_NAMES = ["Background"] + [inv_train[i] for i in range(1, NUM_CLASSES)]
assert len(CLASS_NAMES) == NUM_CLASSES, (
    f"CLASS_NAMES ({len(CLASS_NAMES)}) != NUM_CLASSES ({NUM_CLASSES})"
)

print("CLASS_NAMES by training id:", {i: n for i, n in enumerate(CLASS_NAMES)})
print(f"Model: {YOLO_MODEL}")
print(f"Image size (HxW): {IMAGE_HEIGHT}x{IMAGE_WIDTH}")
print(f"Num classes: {NUM_CLASSES}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}, Early stopping patience: {PATIENCE}")

CLASS_NAMES by training id: {0: 'Background', 1: 'BlackRot', 2: 'Knot', 3: 'Stain'}
Model: yolov8l-seg
Image size (HxW): 384x1024
Num classes: 4
Batch size: 16
Epochs: 100, Early stopping patience: 20


### Create YOLO-Format Dataset Structure

YOLO expects data in a specific folder layout:
```
dataset/
  images/
    train/
    val/
    test/
  labels/
    train/
    val/
    test/
  data.yaml
```

In [4]:
def apply_mask_id_remap(mask):
    """Map on-disk combined-mask class ids -> training ids (see MASK_ID_REMAP in Configuration)."""
    remap = MASK_ID_REMAP if MASK_ID_REMAP else {}
    if not remap:
        return mask

    arr = np.asarray(mask)
    mv = int(arr.max()) if arr.size > 0 else 0
    hi = max(mv, max(remap.keys()) if remap else 0, max(remap.values()) if remap else 0)
    lut = np.arange(hi + 1, dtype=np.int64)
    for src, dst in remap.items():
        src, dst = int(src), int(dst)
        if 0 <= src <= hi:
            lut[src] = dst
    mapped = lut[arr.astype(np.int64)]
    return mapped.astype(arr.dtype, copy=False)


def mask_to_yolo_segments(mask):
    """Convert a single remapped class-id mask (H,W) to YOLOv8-seg polygon label lines."""
    mask = np.asarray(mask)
    if mask.ndim != 2:
        raise ValueError(f"Expected 2D mask, got shape {mask.shape}")

    h, w = mask.shape
    lines = []

    for cls_id in sorted(np.unique(mask)):
        cls_id = int(cls_id)
        if cls_id <= 0:
            continue  # 0 is background in our masks
        cls_idx = cls_id - 1  # YOLO classes are 0-based and do not include background

        bin_mask = (mask == cls_id).astype(np.uint8)
        contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours:
            if contour.shape[0] < 3:
                continue
            contour = contour.squeeze(1)
            if contour.ndim != 2 or contour.shape[0] < 3:
                continue

            # Light simplification keeps files small while preserving shape.
            eps = 0.001 * cv2.arcLength(contour, True)
            contour = cv2.approxPolyDP(contour, eps, True).squeeze(1)
            if contour.ndim != 2 or contour.shape[0] < 3:
                continue

            coords = []
            for x, y in contour:
                xn = min(max((float(x) + 0.5) / w, 0.0), 1.0)
                yn = min(max((float(y) + 0.5) / h, 0.0), 1.0)
                coords.extend([xn, yn])

            if len(coords) >= 6:
                lines.append(f"{cls_idx} " + " ".join(f"{v:.6f}" for v in coords))

    return lines


def dataset_has_yolo_txt_labels(dataset_root):
    """Return True only if train/val/test all have at least one non-empty .txt label."""
    for split in ["train", "val", "test"]:
        labels_dir = Path(dataset_root) / "labels" / split
        txt_files = list(labels_dir.glob("*.txt")) if labels_dir.exists() else []
        if not txt_files:
            return False
        if all(p.stat().st_size == 0 for p in txt_files):
            return False
    return True


# Check if dataset already exists and is already in YOLO-seg .txt format
data_yaml_path_check = YOLO_DATASET_ROOT / "data.yaml"
if data_yaml_path_check.exists() and dataset_has_yolo_txt_labels(YOLO_DATASET_ROOT):
    print(f"YOLO-seg dataset already exists at {YOLO_DATASET_ROOT}")
    print("Skipping dataset creation...")
    data_yaml_path = str(data_yaml_path_check)
else:
    print("Creating YOLO dataset structure and polygon labels...")

    def create_yolo_dataset_structure():
        """Create YOLO-format segmentation dataset with polygon .txt labels."""

        # Clean up old structure if exists (including stale PNG-label dataset).
        if YOLO_DATASET_ROOT.exists():
            shutil.rmtree(YOLO_DATASET_ROOT)

        # Create folders
        for split in ["train", "val", "test"]:
            (YOLO_DATASET_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
            (YOLO_DATASET_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

        # Load split info
        split_file = Path(data_root) / "split.json"
        with open(split_file, "r") as f:
            split_info = json.load(f)

        images_dir = Path(data_root) / "images"
        masks_dir = Path(data_root) / "combined_masks"

        # Copy images and convert masks to YOLO polygon labels for each split
        for split in ["train", "val", "test"]:
            image_ids = split_info[split]
            label_lines_written = 0

            for img_id in image_ids:
                # Copy image
                src_img = images_dir / f"{img_id}_Col.tif"
                dst_img = YOLO_DATASET_ROOT / "images" / split / f"{img_id}.png"

                if src_img.exists():
                    img = Image.open(src_img).convert("RGB")
                    img.save(dst_img)

                # Convert remapped mask to YOLO segment .txt
                src_mask = masks_dir / f"{img_id}_mask.png"
                dst_label = YOLO_DATASET_ROOT / "labels" / split / f"{img_id}.txt"

                label_lines = []
                if src_mask.exists():
                    mask = np.array(Image.open(src_mask))
                    mask = apply_mask_id_remap(mask)
                    label_lines = mask_to_yolo_segments(mask)

                with open(dst_label, "w") as f:
                    if label_lines:
                        f.write("\n".join(label_lines) + "\n")
                        label_lines_written += len(label_lines)

            print(
                f"Prepared {len(image_ids)} samples for {split}: "
                f"{label_lines_written} polygon rows written"
            )

    def create_data_yaml():
        """Create data.yaml for YOLO segmentation training."""
        yolo_nc = NUM_CLASSES - 1  # YOLO classes exclude background
        yolo_names = {i: CLASS_NAMES[i + 1] for i in range(yolo_nc)}
        data_yaml = {
            "path": str(YOLO_DATASET_ROOT),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "nc": yolo_nc,
            "names": yolo_names,
        }

        yaml_path = YOLO_DATASET_ROOT / "data.yaml"
        with open(yaml_path, "w") as f:
            yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

        print(f"Created data.yaml: {yaml_path}")
        return str(yaml_path)

    # Create YOLO dataset structure
    create_yolo_dataset_structure()
    data_yaml_path = create_data_yaml()

print("\nYOLO dataset ready!")


YOLO-seg dataset already exists at /workspaces/Oak-Defect-Detection/data/yolo-format
Skipping dataset creation...

YOLO dataset ready!


### Model Training with Native YOLO API

In [5]:
# Load YOLO model
if TRANSFER_LEARNING and TRANSFER_CHECKPOINT_PATH:
    model = YOLO(TRANSFER_CHECKPOINT_PATH)
    print(f"Loaded transfer learning checkpoint: {TRANSFER_CHECKPOINT_PATH}")
else:
    model = YOLO(f"{YOLO_MODEL}.pt")
    print(f"Loaded pretrained model: {YOLO_MODEL}")

model.to(DEVICE)
print(f"Model on device: {DEVICE}")

Loaded pretrained model: yolov8l-seg
Model on device: cuda:0


In [6]:
def freeze_backbone(model):
    """Freeze backbone parameters, keep head/decoder trainable."""
    freeze_list = [
        "model.0", "model.1", "model.2", "model.3", "model.4",  # backbone
        "model.5", "model.6", "model.7", "model.8",  # neck
    ]
    for name, param in model.model.named_parameters():
        if any(name.startswith(freeze_part) for freeze_part in freeze_list):
            param.requires_grad = False
    print("Backbone frozen")


def unfreeze_backbone(model):
    """Unfreeze all parameters for full fine-tuning."""
    for param in model.model.parameters():
        param.requires_grad = True
    print("Backbone unfrozen")


# Optionally freeze backbone for initial warmup
if FREEZE_BACKBONE and FREEZE_EPOCHS > 0:
    freeze_backbone(model)
    print(f"Will unfreeze after {FREEZE_EPOCHS} epochs")

In [7]:
# Train with YOLO's native API
from torch import cuda

def validate_yolo_seg_labels(dataset_root):
    """Fail fast when segmentation labels are not in YOLO polygon .txt format."""
    for split in ["train", "val", "test"]:
        labels_dir = Path(dataset_root) / "labels" / split
        txt_files = list(labels_dir.glob("*.txt"))
        png_files = list(labels_dir.glob("*.png"))
        non_empty_txt = sum(1 for p in txt_files if p.stat().st_size > 0)

        print(
            f"[{split}] labels: {len(txt_files)} txt ({non_empty_txt} non-empty), "
            f"{len(png_files)} png"
        )

        # Ultralytics segment training expects polygons in .txt files, not mask PNGs.
        if len(txt_files) == 0 and len(png_files) > 0:
            raise RuntimeError(
                f"{labels_dir} contains mask PNG files but no YOLO segment .txt labels. "
                "Convert masks to YOLO polygon labels before training."
            )

        if len(txt_files) > 0 and non_empty_txt == 0:
            raise RuntimeError(
                f"All YOLO label files in {labels_dir} are empty. "
                "Training would treat every sample as background-only."
            )

validate_yolo_seg_labels(YOLO_DATASET_ROOT)

train_kwargs = dict(
    data=data_yaml_path,
    epochs=NUM_EPOCHS,
    imgsz=TRAIN_IMGSZ,
    rect=True,
    batch=BATCH_SIZE,
    workers=NUM_WORKERS,
    device=GPU_ID,
    patience=PATIENCE,
    save=True,
    project=str(MODEL_DIR),
    name="runs",
    lr0=LEARNING_RATE,
    lrf=0.01,  # final LR as fraction of initial
    mosaic=1.0,  # full mosaic augmentation
    flipud=0.5,  # vertical flip
    fliplr=0.5,  # horizontal flip
    hsv_h=0.015,  # HSV hue augmentation
    hsv_s=0.7,  # HSV saturation augmentation
    hsv_v=0.4,  # HSV value augmentation
    degrees=10,  # rotation degrees
    translate=0.1,  # translation fraction
    scale=0.5,  # scale augmentation
    optimizer="SGD",  # optimizer choice (SGD or Adam)
    verbose=True,
)

try:
    results = model.train(**train_kwargs)
except Exception as e:
    oom_tokens = ["out of memory", "acceleratorerror", "cuda error"]
    if any(tok in str(e).lower() for tok in oom_tokens):
        print("\nDetected GPU/pinned-memory issue. Retrying with safer settings...")
        cuda.empty_cache()

        retry_kwargs = dict(train_kwargs)
        retry_kwargs["batch"] = 8 if BATCH_SIZE == -1 else max(1, BATCH_SIZE // 2)
        retry_kwargs["workers"] = max(1, NUM_WORKERS // 2)
        retry_kwargs["mosaic"] = 0.0
        retry_kwargs["mixup"] = 0.0
        retry_kwargs["copy_paste"] = 0.0

        print(
            "Retry params:",
            {k: retry_kwargs[k] for k in ["imgsz", "batch", "workers", "mosaic", "mixup", "copy_paste"]},
        )
        results = model.train(**retry_kwargs)
    else:
        raise

print("\nTraining complete!")
print(f"Best model saved to: {MODEL_DIR / 'runs' / 'weights' / 'best.pt'}")

[train] labels: 688 txt (688 non-empty), 0 png
[val] labels: 132 txt (132 non-empty), 0 png
[test] labels: 205 txt (205 non-empty), 0 png
New https://pypi.org/project/ultralytics/8.4.104 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.103 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspaces/Oak-Defect-Detection/data/yolo-format/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.

In [8]:
# Optional: If backbone was frozen, unfreeze and continue training
if FREEZE_BACKBONE and FREEZE_EPOCHS > 0:
    print("\n" + "="*50)
    print("Unfreezing backbone and continuing training...")
    print("="*50)
    
    # Load best checkpoint from warmup
    best_ckpt = MODEL_DIR / "runs" / "weights" / "best.pt"
    model = YOLO(str(best_ckpt))
    model.to(DEVICE)
    
    # Unfreeze all parameters
    unfreeze_backbone(model)
    
    # Continue training with lower learning rate
    results = model.train(
        data=data_yaml_path,
        epochs=NUM_EPOCHS + FREEZE_EPOCHS,  # continue from where we left off
        imgsz=TRAIN_IMGSZ,
        rect=True,
        batch=BATCH_SIZE,
        device=GPU_ID,
        patience=PATIENCE,
        save=True,
        project=str(MODEL_DIR),
        name="runs_unfrozen",
        lr0=LEARNING_RATE * 0.1,  # lower LR for unfrozen training
        lrf=0.01,
        optimizer="SGD",
        verbose=True,
    )
    
    # Use the unfrozen run's best model
    best_ckpt = MODEL_DIR / "runs_unfrozen" / "weights" / "best.pt"
    print(f"\nFine-tuning complete! Best model: {best_ckpt}")

### Test-Set Evaluation

In [9]:
# Determine which best checkpoint to load
if FREEZE_BACKBONE and FREEZE_EPOCHS > 0:
    best_model_path = MODEL_DIR / "runs_unfrozen" / "weights" / "best.pt"
else:
    best_model_path = MODEL_DIR / "runs" / "weights" / "best.pt"

print(f"Loading best model from: {best_model_path}")
model = YOLO(str(best_model_path))
model.to(DEVICE)
print("Best model loaded!")

Loading best model from: /workspaces/Oak-Defect-Detection/models/STACK_YOLO_L_FullImage/runs/weights/best.pt
Best model loaded!


In [10]:
# Run validation on test set
val_results = model.val(
    data=data_yaml_path,
    split="val",
    imgsz=TRAIN_IMGSZ,
    batch=BATCH_SIZE,
    device=GPU_ID,
    project=str(MODEL_DIR),
    name="runs",
    verbose=True,
)

print("\nTest-set validation complete!")
print(f"mAP50: {val_results.box.map50:.4f}")
print(f"mAP50-95: {val_results.box.map:.4f}")

Ultralytics 8.4.103 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)


YOLOv8l-seg summary (fused): 126 layers, 45,914,201 parameters, 0 gradients, 210.1 GFLOPs
val: Fast image access ✅ (ping: 1.6±0.1 ms, read: 234.5±19.9 MB/s, size: 3207.0 KB)
val: Scanning /workspaces/Oak-Defect-Detection/data/yolo-format/labels/val.cache... 132 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 50.3Mit/s 0.0s


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: Using fork() can cause Polars to deadlock in the child process.
In addition, using fork() with Python in general is a recipe for mysterious
deadlocks and crashes.

The most likely reason you are seeing this error is because you are using the
multiprocessing module on Linux, which uses fork() by default. This will be
fixed in Python 3.14. Until then, you want to use the "spawn" context instead.

See https://docs.pola.rs/user-guide/misc/multiprocessing/ for details.

  self.pid = os.fork()


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.2it/s 4.1s0.4ss
                   all        132        904      0.503      0.292      0.269      0.123      0.472      0.291      0.259      0.103
              BlackRot         51        298      0.537      0.183      0.169     0.0947      0.509      0.178      0.156     0.0667
                  Knot        114        456      0.516      0.539      0.487      0.201      0.489      0.548       0.49      0.179
                 Stain         41        150      0.457      0.153      0.153     0.0727      0.417      0.147      0.132     0.0649
Speed: 0.9ms preprocess, 9.7ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /workspaces/Oak-Defect-Detection/models/STACK_YOLO_L_FullImage/runs-2

Test-set validation complete!
mAP50: 0.2694
mAP50-95: 0.1229


In [11]:
# Save validation metrics to CSV
import csv
import json

# Prepare metrics
metrics = {
    "run_name": METRICS_RUN_NAME,
    "timestamp": RUN_TIMESTAMP,
    "yolo_model": YOLO_MODEL,
    "image_size": f"{IMAGE_HEIGHT}x{IMAGE_WIDTH}",
    "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS,
    "patience": PATIENCE,
    "freeze_backbone": FREEZE_BACKBONE,
    "freeze_epochs": FREEZE_EPOCHS,
    "transfer_learning": TRANSFER_LEARNING,
}

# Add YOLO validation results
if hasattr(val_results, 'box'):
    metrics["map50"] = float(val_results.box.map50)
    metrics["map50_95"] = float(val_results.box.map)
if hasattr(val_results, 'seg'):
    metrics["seg_map50"] = float(val_results.seg.map50) if hasattr(val_results.seg, 'map50') else None
    metrics["seg_map50_95"] = float(val_results.seg.map) if hasattr(val_results.seg, 'map') else None

# Save to JSON
metrics_json_path = METRICS_DIR / f"{METRICS_RUN_NAME}_metrics.json"
with open(metrics_json_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved to: {metrics_json_path}")
print("\n" + "="*50)
print("METRICS SUMMARY")
print("="*50)
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Metrics saved to: /workspaces/Oak-Defect-Detection/data/metrics/stack_yolo_l_fullimage_20260722_164922_metrics.json

METRICS SUMMARY
run_name: stack_yolo_l_fullimage_20260722_164922
timestamp: 20260722_164922
yolo_model: yolov8l-seg
image_size: 384x1024
batch_size: 16
epochs: 100
patience: 20
freeze_backbone: False
freeze_epochs: 5
transfer_learning: False
map50: 0.2694
map50_95: 0.1229
seg_map50: 0.2592
seg_map50_95: 0.1034


In [12]:
# Copy best checkpoint to deployment-ready location
deployment_checkpoint = MODEL_DIR / "best.pt"
if best_model_path.exists():
    shutil.copy(best_model_path, deployment_checkpoint)
    print(f"Best checkpoint copied to: {deployment_checkpoint}")
    print("\nReady for Docker deployment!")
else:
    print(f"Warning: Best checkpoint not found at {best_model_path}")

Best checkpoint copied to: /workspaces/Oak-Defect-Detection/models/STACK_YOLO_L_FullImage/best.pt

Ready for Docker deployment!


### Quick Inference Example

Test the model on a sample image to verify it works correctly.

In [13]:
# Quick inference test on first test image
test_images_dir = YOLO_DATASET_ROOT / "images" / "test"
test_images = list(test_images_dir.glob("*.png"))

if test_images:
    # Run inference on first image
    test_img = test_images[0]
    print(f"Running inference on: {test_img.name}")
    
    results = model.predict(source=str(test_img), conf=0.5)
    
    if results and len(results) > 0:
        result = results[0]
        if hasattr(result, 'masks') and result.masks is not None:
            print(f"Masks shape: {result.masks.shape}")
            print(f"Predictions successful!")
        else:
            print("No masks detected in output")
    else:
        print("No results returned")
else:
    print("No test images found")

Running inference on: 1-29-26_3.29.08.459_Top.png

image 1/1 /workspaces/Oak-Defect-Detection/data/yolo-format/images/test/1-29-26_3.29.08.459_Top.png: 256x1024 1 Stain, 72.8ms
Speed: 2.2ms preprocess, 72.8ms inference, 3.4ms postprocess per image at shape (1, 3, 256, 1024)
Masks shape: torch.Size([1, 256, 1024])
Predictions successful!


In [14]:
print("\n" + "="*70)
print("YOLO SEMANTIC SEGMENTATION TRAINING COMPLETE")
print("="*70)
print(f"\nModel: {YOLO_MODEL}")
print(f"Best checkpoint: {deployment_checkpoint}")
print(f"\nMetrics file: {metrics_json_path}")
print(f"\nTo use this model in Docker:")
print(f"  from ultralytics import YOLO")
print(f"  model = YOLO('{deployment_checkpoint}')")
print(f"  results = model.predict(source='image.jpg')")
print(f"\nAll results saved to: {METRICS_DIR}")
print("="*70)


YOLO SEMANTIC SEGMENTATION TRAINING COMPLETE

Model: yolov8l-seg
Best checkpoint: /workspaces/Oak-Defect-Detection/models/STACK_YOLO_L_FullImage/best.pt

Metrics file: /workspaces/Oak-Defect-Detection/data/metrics/stack_yolo_l_fullimage_20260722_164922_metrics.json

To use this model in Docker:
  from ultralytics import YOLO
  model = YOLO('/workspaces/Oak-Defect-Detection/models/STACK_YOLO_L_FullImage/best.pt')
  results = model.predict(source='image.jpg')

All results saved to: /workspaces/Oak-Defect-Detection/data/metrics


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
notebooks_dir = project_root / "notebooks"
if str(notebooks_dir) not in sys.path:
    sys.path.append(str(notebooks_dir))

from functions.metrics_export import append_row_to_csv, build_run_row, first_available, infer_split_sizes

local_vars = locals()
train_split_size, val_split_size, test_split_size = infer_split_sizes(project_root)

metrics_dir = Path(local_vars.get("METRICS_DIR", project_root / "data" / "metrics"))
csv_path = metrics_dir / "TRAINING_RESULTS_SUMMARY.csv"

extra_training_params = {}
for key in [
    "TRAIN_IMGSZ",
    "PATIENCE",
    "FREEZE_BACKBONE",
    "FREEZE_EPOCHS",
    "TRANSFER_LEARNING",
    "YOLO_MODEL",
    "GPU_ID",
]:
    if key in local_vars:
        extra_training_params[key] = local_vars[key]

if "train_kwargs" in local_vars and isinstance(local_vars["train_kwargs"], dict):
    for key, value in local_vars["train_kwargs"].items():
        extra_training_params[f"train_kwargs_{key}"] = value

row = build_run_row(
    notebook_name="YOLO_FullImage",
    model_name=str(first_available(local_vars, ["YOLO_MODEL"]) or "YOLO-seg"),
    raw_data_folder=first_available(local_vars, ["RAW_DIR"]),
    image_height=first_available(local_vars, ["IMAGE_HEIGHT"]),
    image_width=first_available(local_vars, ["IMAGE_WIDTH"]),
    batch_size=first_available(local_vars, ["BATCH_SIZE"]),
    num_workers=first_available(local_vars, ["NUM_WORKERS"]),
    num_epochs_config=first_available(local_vars, ["NUM_EPOCHS"]),
    num_epochs_actual=first_available(local_vars, ["NUM_EPOCHS"]),
    optimizer_name=first_available(local_vars, ["OPTIMIZER"]) or "SGD",
    learning_rate=first_available(local_vars, ["LEARNING_RATE"]),
    patience=first_available(local_vars, ["PATIENCE"]),
    num_classes=first_available(local_vars, ["NUM_CLASSES"]),
    class_names=first_available(local_vars, ["CLASS_NAMES"]),
    train_split_size=train_split_size,
    val_split_size=val_split_size,
    test_split_size=test_split_size,
    accuracy=(float(val_results.seg.map50) if "val_results" in local_vars and hasattr(val_results, "seg") and hasattr(val_results.seg, "map50") else None),
    mean_iou=(float(val_results.seg.map) if "val_results" in local_vars and hasattr(val_results, "seg") and hasattr(val_results.seg, "map") else None),
    checkpoint_path=first_available(local_vars, ["deployment_checkpoint", "best_model_path"]),
    onnx_path=None,
    run_name=first_available(local_vars, ["METRICS_RUN_NAME"]),
    transfer_learning=first_available(local_vars, ["TRANSFER_LEARNING"]),
    transfer_checkpoint_path=first_available(local_vars, ["TRANSFER_CHECKPOINT_PATH"]),
    freeze_encoder_epochs=first_available(local_vars, ["FREEZE_EPOCHS"]),
    optuna_enabled=False,
    extra_training_params=extra_training_params,
    extra_metrics={
        "box_map50": float(val_results.box.map50) if "val_results" in local_vars and hasattr(val_results, "box") else None,
        "box_map50_95": float(val_results.box.map) if "val_results" in local_vars and hasattr(val_results, "box") else None,
        "seg_map50": float(val_results.seg.map50) if "val_results" in local_vars and hasattr(val_results, "seg") and hasattr(val_results.seg, "map50") else None,
        "seg_map50_95": float(val_results.seg.map) if "val_results" in local_vars and hasattr(val_results, "seg") and hasattr(val_results.seg, "map") else None,
    },
)

written_path = append_row_to_csv(csv_path, row)
print(f"Appended metrics row to: {written_path}")